# Apple Silicon TorchANI Efficiency Profile

This notebook isolates the performance question: which parts of the TorchANI/ANI path run well on Apple Silicon, and which parts stay CPU-bound or become expensive on MPS? It profiles the same local `vendor/torchani` checkout and the same `mod_ani` model builders used by the proposal workflow.

Run the cells top to bottom. The default uses TorchANI `TestData` so it finishes quickly; flip `PROFILE_ANI1X = True` when you want a larger, more representative batch.

In [ ]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

for module_name in list(sys.modules):
    if module_name == "mod_ani" or module_name.startswith("mod_ani."):
        del sys.modules[module_name]

from mod_ani.local_torchani import use_local_torchani
TORCHANI_SOURCE = use_local_torchani()

import torch
import torchani

print(f"Project root: {PROJECT_ROOT}")
print(f"TorchANI source: {TORCHANI_SOURCE}")
print(f"TorchANI imported from: {Path(torchani.__file__).resolve()}")
print(f"PyTorch: {torch.__version__}")
print(f"MPS available: {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Choose What To Profile

Keep the default first. Once the notebook is working, increase `BATCH_SIZE`, switch `MODEL_KIND`, or set `PROFILE_ANI1X = True`.

In [ ]:
from mod_ani.config import ani1x_config, quick_test_config

PROFILE_ANI1X = False
MODEL_KIND = "electron_radial"  # "electron_radial" or "baseline"
BATCH_SIZE = 128
WARMUP = 2
REPEATS = 5

if PROFILE_ANI1X:
    config = ani1x_config(
        batch_size=max(BATCH_SIZE, 512),
        limit_train_batches=1,
        limit_valid_batches=1,
        cache_batches=False,
        verbose=True,
    )
else:
    config = quick_test_config(
        batch_size=BATCH_SIZE,
        limit_train_batches=1,
        limit_valid_batches=1,
        cache_batches=True,
        verbose=True,
    )

config.model_kind = MODEL_KIND
config.max_epochs = 1
config.refresh_batches = False
config.dtype = "float32"

print(config)
print(f"Resolved default device: {config.resolved_device()}")

## 2. Download And Prepare One Batch

This uses the same dataset downloader and batch preparation code as the main proposal notebook. Batches are reused unless `config.refresh_batches = True`.

In [ ]:
from mod_ani.data import describe_dataset, download_dataset, prepare_batched_dataset
from mod_ani.profiling import first_batch

print("Opening dataset...")
dataset = download_dataset(config)
print(describe_dataset(dataset))

print("Preparing batches...")
batched = prepare_batched_dataset(dataset, config)

raw_batch = first_batch(batched, split="training")
print("First training batch tensor shapes:")
for key, value in raw_batch.items():
    print(f"  {key:12s} shape={tuple(value.shape)} dtype={value.dtype} device={value.device}")

## 3. Inspect The Model

This builds the selected model from the local source tree. The timing cells rebuild a fresh copy for each device so the training-step timing does not contaminate later measurements.

In [ ]:
from mod_ani.models import build_model, count_parameters

model = build_model(config)
aev = model.aev_computer
print(model)
print(f"Trainable parameters: {count_parameters(model):,}")
print(f"AEV strategy: {aev.strategy}")
print(f"AEV output dimension: {aev.out_dim}")
print(f"Radial features: {aev.radial_len}; angular features: {aev.angular_len}")
print(f"Neighborlist: {aev.neighborlist.__class__.__name__}")

## 4. Profile CPU, MPS, And CUDA If Available

The table times the major pieces of the pure PyTorch AEV path plus one full training step. `pairs` and `triples` show how much combinatorial work was present in the selected batch.

In [ ]:
import copy
import pandas as pd

from mod_ani.profiling import available_devices, profile_aev_and_training_step

all_rows = []
devices = available_devices()
print(f"Devices to profile: {devices}")

for device_name in devices:
    print("=" * 88)
    print(f"Profiling device: {device_name}")
    device_config = copy.deepcopy(config)
    device_config.device = device_name
    device_model = build_model(device_config)
    try:
        rows = profile_aev_and_training_step(
            device_model,
            raw_batch,
            device_config,
            device_name=device_name,
            warmup=WARMUP,
            repeats=REPEATS,
        )
        all_rows.extend(rows)
        for row in rows:
            print(
                f"{row['operation']:22s} mean={row['mean_ms']:10.3f} ms "
                f"median={row['median_ms']:10.3f} ms pairs={row['pairs']} triples={row['triples']}"
            )
    except Exception as exc:
        print(f"FAILED on {device_name}: {type(exc).__name__}: {exc}")
        all_rows.append({
            "operation": "profile_failed",
            "device": device_name,
            "error": f"{type(exc).__name__}: {exc}",
        })

df = pd.DataFrame(all_rows)
display(df)

## 5. Save And Plot Results

The CSV goes under `outputs/profiling/`, which is meant for generated run artifacts.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = PROJECT_ROOT / "outputs" / "profiling"
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "apple_efficiency_profile.csv"
df.to_csv(csv_path, index=False)
print(f"Saved profile table: {csv_path}")

plot_df = df[df["operation"].ne("profile_failed")].copy()
plot_df = plot_df.sort_values(["operation", "device"])

fig, ax = plt.subplots(figsize=(12, 6))
for device_name, group in plot_df.groupby("device"):
    ax.bar(
        group["operation"] + "
" + device_name,
        group["mean_ms"],
        label=device_name,
        alpha=0.8,
    )
ax.set_ylabel("Mean time (ms)")
ax.set_yscale("log")
ax.set_title("TorchANI operation timings by device")
ax.tick_params(axis="x", rotation=75)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## 6. Quick Interpretation

This cell gives a first-pass answer about whether larger batches or Apple-specific rewrites are likely to help.

In [ ]:
usable = df[df["operation"].ne("profile_failed")].copy()
if usable.empty:
    print("No successful profile rows to interpret.")
else:
    train_rows = usable[usable["operation"].eq("single_train_step")].sort_values("mean_ms")
    if not train_rows.empty:
        fastest = train_rows.iloc[0]
        print(
            "Fastest measured training step: "
            f"{fastest['device']} at {fastest['mean_ms']:.3f} ms "
            f"for batch_size={int(fastest['batch_size'])}, atoms/conformer={int(fastest['atoms_per_conformer'])}."
        )

    for device_name, group in usable.groupby("device"):
        slowest = group.sort_values("mean_ms", ascending=False).iloc[0]
        print(
            f"Largest timed cost on {device_name}: {slowest['operation']} "
            f"({slowest['mean_ms']:.3f} ms)."
        )

    if set(train_rows["device"]) >= {"cpu", "mps"}:
        cpu_ms = float(train_rows[train_rows["device"].eq("cpu")]["mean_ms"].iloc[0])
        mps_ms = float(train_rows[train_rows["device"].eq("mps")]["mean_ms"].iloc[0])
        ratio = mps_ms / cpu_ms
        print(f"MPS/CPU train-step ratio: {ratio:.2f}x")
        if ratio > 1.0:
            print(
                "For this batch, MPS is slower. That usually means the AEV path is dominated by "
                "indexing, sorting, masking, triple construction, or synchronization overhead rather "
                "than dense neural-network math."
            )
        else:
            print(
                "For this batch, MPS is faster. Larger batches may help further if the bottleneck is "
                "full_model_forward or single_train_step rather than neighbors_to_triples."
            )